# Rust 116: Concurrency and Async/Await

Chapter 16, and the wall from chapter 9 comes down. Everything so far ran on one thread; from here the language has opinions about **many**. This is the pivot of the applied track — chapters 17–19 (HTTP, Redis, PostgreSQL) assume everything built here, so the chapter teaches two worlds and the bridge between them: **OS threads** with `Send`/`Sync` as the compile-time contract that makes them safe, and **async/await** — not threads at all, but suspended *stack frames* driven by a runtime — with `tokio` as the de-facto standard runtime. The thesis: Rust's edge is that both worlds' deadliest bugs (data races, forgetting to join, use-after-free across an await) are **compile errors**, and the chapter's job is to show you the error texts until they feel like old friends.

Everything here is a **runnable example**. Read the markdown, run the cell below it, and change nothing — the practice twin of this notebook is `16_concurrency-and-async_p.ipynb`.

## Who this is written for

You are assumed to be a competent programmer arriving from Python and C++. Python's `threading` module and `asyncio` are old friends; C++ taught you `std::thread` and the smell of a data race in production. What is new in Rust: **no GIL** — two threads really do execute your code at once, so the borrow checker's `Send`/`Sync` marker traits are the only thing standing between you and corruption (and they stand there at *compile time*); `async fn` is not a green thread but a compiler-generated **state machine** whose size you can print; and there is **no async in std** — the runtime (wakers, the scheduler, timers) is a library, and `tokio` is the one chapters 17–19 build on.

## How to read this notebook

- Markdown cells explain; code cells demonstrate. **Run each code cell as you reach it.** Cells are independent — no sandbox, no files touched, so restart-and-run-from-the-top always works.
- Code that intentionally *fails* is shown as a comment, together with the diagnostic you would get. §11 is the catalogue.
- The running domain is the trading desk: a price feed fanning out to workers, a shared position ledger, a slow upstream that must not stall a fast one. Every primitive here earns its place on that domain.
- One honesty note runs through the chapter: cells print *timings* (§6, §9) that will differ on your machine. The **asserts** pin down what is deterministic; the prose says what is not.


### Contents

- [0. Notebook Conventions: Threads and Runtimes in a Notebook](#0-notebook-conventions-threads-and-runtimes-in-a-notebook)
- [1. OS Threads: `spawn`, `join`, `Builder`](#1-os-threads-spawn-join-builder)
- [2. `Send` and `Sync`: The Compile-Time Race Proof](#2-send-and-sync-the-compile-time-race-proof)
- [3. Channels: Ownership as the Message Protocol](#3-channels-ownership-as-the-message-protocol)
- [4. `Mutex` and `RwLock`: Shared State Under Guard](#4-mutex-and-rwlock-shared-state-under-guard)
- [5. Atomics: Lock-Free Counters and Flags](#5-atomics-lock-free-counters-and-flags)
- [6. The Problem Threads Solve Poorly](#6-the-problem-threads-solve-poorly)
- [7. `async`/`await`: Futures Are Frames](#7-asyncawait-futures-are-frames)
- [8. The Runtime: `tokio`, `spawn`, `JoinHandle`](#8-the-runtime-tokio-spawn-joinhandle)
- [9. `select!` and `timeout`: Racing Futures](#9-select-and-timeout-racing-futures)
- [10. Blocking vs Async: The Rules of the Road](#10-blocking-vs-async-the-rules-of-the-road)
- [11. Reading the Compiler: E-Codes](#11-reading-the-compiler-e-codes)
- [12. Mental Model: The Receipts](#12-mental-model-the-receipts)
- [13. Cheat Sheet](#13-cheat-sheet)
- [14. Review](#14-review)

If you are returning to this chapter later, §13 is the one-page summary, §11 the diagnostics catalogue, and §10 the judgment call the whole chapter exists to train.


## 0. Notebook Conventions: Threads and Runtimes in a Notebook

This notebook runs on the **evcxr** kernel, which wraps each cell into a generated program and compiles it incrementally. Three consequences matter for every cell below:

- **There is no `fn main`.** The kernel supplies one; you are typing its body.
- **State persists across cells.** A `let` from cell 2 is still live in cell 20 — including `use` statements.
- **Items are order-independent, statements are not.** Two `fn`s may call each other regardless of order; two `let`s may not use each other's names before they run.

There is **one namespace per notebook**. Every struct and function name below is unique across the notebook. Chapter 09's `:dep` magic is back for one crate — and the first cell that needs a name does that crate's `use`; later cells rely on persistence (a repeated `use` is E0252).

Two chapter-specific conventions:

- **A cell that wants async creates its runtime, uses it, and moves on.** `let rt = tokio::runtime::Runtime::new().unwrap();` at the top of a cell, `rt.block_on(async { ... })` inside it. That is the manual form of what `#[tokio::main]` writes *for* you in a binary — and in a notebook (or a test), the manual form is the only form, because the kernel owns `main`.
- **Spawned threads outlive no cell cleanly.** Every `thread::spawn` in this notebook is paired with a `join()` before the cell ends, so each cell is deterministic on its own. Output *between* two threads is scheduler-ordered; the asserts only pin what is guaranteed.


In [ ]:
// 0.1 The chapter's crate cell: run once per kernel session.
:dep tokio = { version = "1", features = ["full"] }

let rt = tokio::runtime::Runtime::new().unwrap();
let answer = rt.block_on(async { 40 + 2 });
assert_eq!(answer, 42);
println!("tokio loaded; block_on smoke: {}", answer);


`features = ["full"]` is the everything-bagel — macros, the multi-thread scheduler, timers, sync primitives, networking. Chapters 17–19 will keep it and stop thinking about it; the feature-flag discipline of chapter 15 (`uuid` without `v4` compiles `new_v4` *out*) applies here too: `rt-multi-thread` is the scheduler, `macros` is `#[tokio::main]` and `select!`, `time` is `sleep` and `timeout`. The cell ran `block_on` — the one method you must own before §8: *drive this future to completion, on this thread, now.*

## 1. OS Threads: `spawn`, `join`, `Builder`

`thread::spawn` takes a closure and hands it to the operating system: a real thread, really parallel, preemptively scheduled — Python's GIL does not exist here, and nothing about that sentence is unsafe *by construction*, because the closure's captures are checked (§2). The spawn returns a **`JoinHandle<T>`**; `join()` blocks until the thread finishes and hands back `Result<T>` — the thread's return value, or the panic it died with. A thread whose handle you drop keeps running (detached), which is exactly the "fire and forget" that this notebook's cells avoid: every thread gets joined, every cell deterministic. `Builder` names threads (debuggers and panic messages read it) and sizes stacks; `available_parallelism` reports what the OS thinks "parallel" means.


In [ ]:
// 1.1 Spawn, join, name: a thread is a function call that happens elsewhere.
use std::thread;

let handle = thread::spawn(|| 21 * 2);
// join blocks, then hands back the closure's return value as Result
assert_eq!(handle.join().unwrap(), 42);

// Builder: name it, then spawn — the name shows up in panic messages
let named = thread::Builder::new()
    .name("feed-worker".into())
    .spawn(|| thread::current().name().unwrap_or("?").to_string())
    .unwrap();
assert_eq!(named.join().unwrap(), "feed-worker");

// how much hardware is the OS willing to give us?
println!(
    "spawn/join works; available_parallelism = {}",
    thread::available_parallelism().map(|n| n.get()).unwrap_or(0)
);


Three reads on that cell. **`join()` returns `Result` because the thread can panic** — `unwrap()` here is the *happy-path assumption made explicit*, and chapter 11's panic-payloading rules apply verbatim across the join. **The named thread came back with its name**: a closure can ask `thread::current()` about itself, which is how logs on a 32-thread server stay attributable. **`available_parallelism` is a hint, not a limit** — the OS happily multiplexes a thousand threads over those cores; what it cannot do is make a thousand *blocked-on-I/O* threads cheap, which is the crack §6 drives a truck through.


## 2. `Send` and `Sync`: The Compile-Time Race Proof

Two auto traits are the entire safety story: **`Send`** — *it is safe to move this value to another thread* — and **`Sync`** — *it is safe to share `&T` across threads* (which holds exactly when `T: Send` for its internals; formally, `T: Sync` ⇔ `&T: Send`). They are **marker traits**: no methods, checked by the compiler from the types involved. The consequences are absolute. `Rc<i32>` is neither (its reference count is a plain non-atomic increment — two threads would corrupt it), so `thread::spawn` with an `Rc` capture is a compile error — **E0277: `` `Rc<i32>` cannot be sent between threads safely ``, with the note `required because it appears within type Rc<i32>`**. The fix is spelled by the error itself: `Arc<T>`, the atomic-counted twin from chapter 14. `RefCell` is `Send` but not `Sync` (its borrow flag is likewise non-atomic); `Mutex<T>` is both, *because it guards* — chapter 14's `Arc<RefCell<i32>>` refused by `thread::spawn` with the suggestion "use `std::sync::RwLock` instead" was this chapter waving from across the wall.

The escape hatch for non-`'static` data is **`thread::scope`**: threads spawned inside the scope may borrow *anything that outlives the scope*, and the compiler enforces the discipline — all scope borrows end when the scope does (it joins for you), and code inside cannot out-borrow that moment.


In [ ]:
// 2.1 The auto traits at work: Arc crosses threads, Rc is refused at compile time.
use std::sync::Arc;

let shared = Arc::new(vec![1, 2, 3]);
let handle = thread::spawn(move || shared.iter().sum::<i32>());
assert_eq!(handle.join().unwrap(), 6);

// what the compiler does to Rc across a spawn (the code, as a comment):
//   let rc = std::rc::Rc::new(5);
//   thread::spawn(move || println!("{}", rc));
//   error[E0277]: `Rc<i32>` cannot be sent between threads safely
//     note: required because it appears within type `Rc<i32>`
//     help: ...use `std::sync::Arc<T>` instead
// and RefCell is Send-but-not-Sync: one thread may OWN it, none may SHARE it
fn assert_send<T: Send>() {}
fn assert_sync<T: Sync>() {}
assert_send::<std::cell::RefCell<i32>>();
assert_sync::<Arc<std::sync::Mutex<i32>>>();
// assert_sync::<std::cell::RefCell<i32>>();  // E0277: `RefCell<i32>` cannot be shared between threads safely
println!("Arc crossed a thread; Rc/RefCell-refusals are E0277 at compile time");


In [ ]:
// 2.2 thread::scope: borrow non-'static data on threads — the borrow ends with the scope.
let mut nums = vec![1, 2, 3, 4];
thread::scope(|s| {
    // every spawn may borrow `nums` because the SCOPE outlives the threads
    for chunk in nums.chunks_mut(2) {
        s.spawn(move || {
            for v in chunk.iter_mut() { *v *= 10; }
        });
    }
    // (scope ends here: implicit joins — and with them the borrows)
});
// safe: the scope promised the threads are done before this line
assert_eq!(nums, vec![10, 20, 30, 40]);
println!("scope: {:?} mutated on borrowed threads, sum {}", nums, nums.iter().sum::<i32>());


Three reads on those cells. **The E0277 refusal is a proof, not a warning**: the compiler examined `Rc`'s reference count, saw a non-atomic increment, and refused — no test suite catches it later because it never compiled. **The `assert_send`/`assert_sync` idiom is a zero-cost unit test for types**: one line pins a type's thread-safety story, and chapter 14's `Arc<RefCell<i32>>` passes both *because* `Mutex` does the guarding. **`scope`'s discipline is the lifetime system applied to threads**: the E0502 you get for reading `nums` *inside* the scope while chunk-borrows are live is the same shape as any overlapping-borrow error — the scope's implicit join is just a borrow region. Rule: **`spawn` for `'static`/owned work, `scope` for borrowed fan-out over a slice you own.**

## 3. Channels: Ownership as the Message Protocol

A channel is a queue with compile-time semantics: `mpsc::channel()` gives a **`Sender`** and a **`Receiver`**, multiple producers (clone the `Sender`), one consumer. The design move is that **`send` takes ownership**: the value you send is *moved* — the compiler will not let you touch it afterwards, so "racing on the sent value" is unrepresentable. `send` returns `Result` (a channel whose receiver dropped has no future: `SendError`); `recv` returns `Result` (all senders dropped and queue empty: `RecvError`). Dropping the last `Sender` **closes** the channel — the receiver's `for` loop ends; that is how a worker pool learns "no more work" without a sentinel message. `sync_channel(0)` is a **rendezvous** channel: zero buffered slots, `send` blocks until a receiver takes the value hand-to-hand — synchronous transmission as a type.


In [ ]:
// 3.1 mpsc: move semantics make the race impossible; drop closes the channel.
use std::sync::mpsc;

let (tx, rx) = mpsc::channel::<i32>();
for i in 0..4 {
    let tx = tx.clone();
    thread::spawn(move || tx.send(i * i).unwrap());
}
drop(tx); // our own sender goes; the channel closes when the LAST one does
let mut squares: Vec<i32> = rx.iter().collect(); // iter() ends when the channel closes
squares.sort(); // arrival order is scheduler-ordered; the SET is guaranteed
assert_eq!(squares, vec![0, 1, 4, 9]);
println!("mpsc fan-in: {:?}", squares);


In [ ]:
// 3.2 The error kinds and the rendezvous channel: blocking as an API.
use std::sync::mpsc::{sync_channel, RecvError, SendError};

// send after the receiver is dropped: the message has nowhere to go
let (tx, rx) = mpsc::channel::<i32>();
drop(rx);
assert!(matches!(tx.send(1), Err(SendError(_))));
// recv after all senders dropped (and the queue is empty): done, not error-polling
let (_, rx2) = mpsc::channel::<i32>();
assert!(matches!(rx2.recv(), Err(RecvError { .. })));
// sync_channel(0): send blocks until recv takes it — a hand-to-hand transfer
let (stx, srx) = sync_channel::<i32>(0);
let h = thread::spawn(move || stx.send(7).unwrap());
assert_eq!(srx.recv().unwrap(), 7); // recv FIRST: send(0) cannot complete alone
assert_eq!(h.join().unwrap(), ());
println!("SendError/RecvError kinds + rendezvous(0) verified");


Three reads on those cells. **`rx.iter().collect()` after `drop(tx)` is the whole producer/consumer pattern in two lines** — close-then-drain, no sentinel, no sleep-poll; the four squares arrive in *any* order (scheduler), so the cell sorts before asserting: pin the guarantee, not the timing. **The error kinds are lifecycle, not failure**: `SendError` means "the consumer is gone" (time to shut the thread down), `RecvError` means "no producer will ever speak again" (time to return); threads that treat either as anything other than a shutdown signal are the classic bug. **`sync_channel(0)` makes blocking the *point*** — the sender's `send` cannot return until the value is received, which is the strictest backpressure a channel can express; capacity `N` generalizes it to "at most N in flight".


## 4. `Mutex` and `RwLock`: Shared State Under Guard

When threads must *mutate* shared data, the type system offers the guarded forms: **`Mutex<T>`** — one key at a time — and **`RwLock<T>`** — any number of readers *or* one writer. The design move is that the lock is a property of the **data**: you cannot touch the `T` without going through `lock()`, and `lock()` hands you a **guard** whose `Deref`/`DerefMut` (chapter 14) exposes the contents — and whose **`Drop` releases the lock**. Unlocking is RAII, so "forgot to release" is unrepresentable. The standard composite is `Arc<Mutex<T>>`: the `Arc` shares it, the `Mutex` guards it — chapter 14's `Arc<RefCell<T>>` was the single-threaded rehearsal, and its thread-safety refusal (§2) is why the step up exists.

Two sharp edges get their own cell: **poisoning** (a panic while holding the lock marks it poisoned; every later `lock()` returns `Err(PoisonError)` wrapping the guard) — and the trap our own probe hit: a `PoisonError` you *bind* with `let` keeps the guard alive, and the next `lock()` **deadlocks**. Inspect results inline; recover with `into_inner()`.


In [ ]:
// 4.1 Arc<Mutex<T>>: eight threads, one counter, zero races.
use std::sync::{Mutex, RwLock};

let counter = Arc::new(Mutex::new(0u64));
let mut handles = Vec::new();
for _ in 0..8 {
    let counter = Arc::clone(&counter);
    handles.push(thread::spawn(move || {
        for _ in 0..100 {
            *counter.lock().unwrap() += 1; // guard locks on entry, unlocks on drop
        }
    }));
}
for h in handles { h.join().unwrap(); }
assert_eq!(*counter.lock().unwrap(), 800);
println!("8 threads x 100 increments through one Mutex: {}", counter.lock().unwrap());


In [ ]:
// 4.2 Poisoning: panic-while-holding marks the lock; recovery is explicit.
let ledger = Arc::new(Mutex::new(100u32));
let ledger2 = Arc::clone(&ledger);
let _ = thread::spawn(move || {
    let _g = ledger2.lock().unwrap();
    panic!("crashed mid-update, guard still held");
})
.join(); // join returns Err(panic payload); the LOCK is now poisoned

// inspect INLINE: binding the Err(PoisonError) with `let` keeps its guard alive
// and the NEXT lock() would deadlock — the trap this chapter's author probed.
assert!(ledger.lock().is_err());
// recovery: into_inner() hands back the data, poison ignored (value was left mid-update)
let recovered = *ledger.lock().unwrap_or_else(|p| p.into_inner());
assert_eq!(recovered, 100);
println!("poisoned lock refuses unwrap; into_inner recovered: {}", recovered);


In [ ]:
// 4.3 RwLock: many readers at once, one writer, never both.
let book = RwLock::new(vec![1, 2, 3]);
{
    let r1 = book.read().unwrap();
    let r2 = book.read().unwrap(); // a second reader: fine
    assert_eq!((r1[0], r2.len()), (1, 3));
} // guards drop, reader slots free
*book.write().unwrap() = vec![9]; // writer takes it exclusively
assert_eq!(book.read().unwrap()[0], 9);
println!("RwLock: 2 concurrent reads then an exclusive write, all RAII");


Three reads on those cells. **800/800 with no lost updates** is `Mutex` doing the one thing `Arc` cannot alone — `+=` on an `Arc<i32>` would be a data race the compiler refuses (E0599: no `AtomicUsize`-style mutation through a shared reference); the guard *is* the proof of exclusive access. **Poisoning is the panic-propagation story continued across a join**: the data was left mid-update, so Rust makes every later accessor *acknowledge* the break — `unwrap_or_else(|p| p.into_inner())` is the sentence "I looked at the wreck and am continuing anyway" written in code. **`RwLock`'s reads compose**: two `read()` guards live at once is legal and cheap; the *writer's* exclusivity is what a plain `&mut` would demand, granted only through the lock.


## 5. Atomics: Lock-Free Counters and Flags

For single machine-word state, `std::sync::atomic` offers types that need no lock: **`AtomicUsize`/`AtomicI64`/`AtomicBool`**, all `Send + Sync` by definition, their operations indivisible. `fetch_add` returns the **previous** value (the increment and the read are one operation); `swap` exchanges and returns the old; **`compare_exchange`** is the primitive every lock-free algorithm is built from: *write `new` only if the current value is still `expected`* — returning `Result` so the loser knows, atomically. The `Ordering` argument names the memory-ordering contract: `Relaxed` (atomic alone, no other guarantees — right for plain counters), `Acquire`/`Release` (pair them and everything written before a `Release` store is visible after the matching `Acquire` load — the publish pattern), `SeqCst` (the single total order — the "don't make me think" default). This chapter uses `SeqCst` for correctness-shaped examples and `Relaxed` for counters; §14's review returns to why choosing *less* is an optimization you must be able to defend.


In [ ]:
// 5.1 fetch_add, swap, and the compare_exchange loop.
use std::sync::atomic::{AtomicUsize, Ordering};

let hits = Arc::new(AtomicUsize::new(0));
let mut handles = Vec::new();
for _ in 0..4 {
    let hits = Arc::clone(&hits);
    handles.push(thread::spawn(move || {
        for _ in 0..100 {
            hits.fetch_add(1, Ordering::Relaxed); // atomic counter: Relaxed is correct
        }
    }));
}
for h in handles { h.join().unwrap(); }
assert_eq!(hits.load(Ordering::Relaxed), 400);

// fetch_add RETURNS the previous value — read and write in one operation
let seq = AtomicUsize::new(10);
assert_eq!(seq.fetch_add(5, Ordering::SeqCst), 10);
assert_eq!(seq.load(Ordering::SeqCst), 15);
assert_eq!(seq.swap(0, Ordering::SeqCst), 15);

// compare_exchange: the CAS loser finds out atomically
let cell = AtomicUsize::new(100);
assert_eq!(cell.compare_exchange(100, 200, Ordering::SeqCst, Ordering::SeqCst), Ok(100));
let losing = cell.compare_exchange(100, 999, Ordering::SeqCst, Ordering::SeqCst);
assert_eq!(losing, Err(200)); // current value handed back: 100 is stale
println!("atomics: 4x100 lock-free increments = {}; CAS loser sees Err(current)", hits.load(Ordering::Relaxed));


The CAS cell is lock-free programming in four lines: the first `compare_exchange` succeeded because reality still matched the expectation; the second *failed and received the current value* — which is not an error to retry blindly but information: "your assumption aged out between your load and your store." Every lock-free queue and every `OnceLock` under the hood is this loop with different decorations. Note what the counter cell does *not* need: no `Mutex`, no guard, no poison — one machine word with an indivisible increment is the whole story, which is why hot-path metrics in real services are `AtomicU64`s. The cost is honest: `Ordering` arguments push memory-model thinking into your signatures, and §10 will show the async world politely declining that complexity for anything bigger than a word.


## 6. The Problem Threads Solve Poorly

Everything through §5 is real parallelism: four cores doing four computations. Now point those threads at the workloads servers actually have — *waiting*. A thread blocked on I/O holds a stack (MBs, reserved), kernel scheduler state, and a context switch's worth of cost, and contributes **zero computation** while it waits. The classic architecture — one thread per connection — survives hundreds of connections and collapses beyond thousands. The cell below builds the honest minimal version: 10,000 waiters, each sleeping 10ms. Nothing is being computed; the only work is *bookkeeping the waiting*.

Watch what the numbers say. This is a wall-clock cell — your milliseconds **will** differ. What does not differ: at 10,000 *compute* tasks both columns would be equal (the cores are the limit); at 10,000 *waiters* the gap is structural and grows with N.


In [ ]:
// 6.1 10,000 sleepers, both ways. Timings vary by machine and load — read the RATIO.
const N: u32 = 10_000;

// way one: an OS thread per waiter
let t0 = std::time::Instant::now();
let handles: Vec<_> = (0..N)
    .map(|_| thread::spawn(|| thread::sleep(std::time::Duration::from_millis(10))))
    .collect();
for h in handles { h.join().unwrap(); }
let threads_ms = t0.elapsed().as_millis();

// way two: a tokio task per waiter — the runtime multiplexes them over a few threads
let rt = tokio::runtime::Runtime::new().unwrap();
let t0 = std::time::Instant::now();
rt.block_on(async {
    let mut hs = Vec::new();
    for _ in 0..N {
        hs.push(tokio::spawn(async {
            tokio::time::sleep(std::time::Duration::from_millis(10)).await;
        }));
    }
    for h in hs { h.await.unwrap(); }
});
let tasks_ms = t0.elapsed().as_millis();
println!(
    "10,000 waiters x 10ms: threads={}ms tasks={}ms (ratio ~{}x — yours will differ)",
    threads_ms, tasks_ms, threads_ms / tasks_ms.max(1)
);


On this chapter's machine across several runs: **threads ~260–290ms, tasks ~25–35ms** — a factor of ~8–11× that widens as N grows, because 10,000 threads is 10,000 stacks and 10,000 scheduler entries, while 10,000 tasks is a few kilobytes each on a heap plus a handful of worker threads doing epoll. Read the numbers with discipline, though: the *thread* column includes page-faulting ten thousand stacks; a pooled-thread design narrows the gap by reusing threads — and still pays at scale, because a *blocked* thread cannot be reused. That is the crack async walks into: **concurrency for the price of frames, not threads**. What async *is* — and what a `Future` physically is — is next; the answer is stranger and smaller than "a green thread": it is a suspended stack frame with a compile-time-known size.

## 7. `async`/`await`: Futures Are Frames

`async fn slow(a: A, b: B) -> T` desugars to a plain `fn` returning an **`impl Future<Output = T>`** — and *calling* it runs **no code at all**. It constructs a value: the future, a compiler-generated **state machine** holding the arguments and every local that must survive across an `await`. The machine has one state per suspension point; `.await` is (conceptually) the transition: **poll** the inner future, and if it is not ready, *return from the outer function entirely* — keeping everything it will need on restart inside the future. Nothing is suspended on a stack; there is no thread behind it; the frame is a **value on the heap** with a size you can print, like any struct. That is the sentence to internalize: **async is not threads with better PR — it is `enum`-shaped control flow**, and a future that never gets `.await`ed (or polled by a runtime) is a value that does nothing — which is why "forget the `.await`" produces no error, only silence.

`async` blocks are the same machine in literal form: `async { ... }` is a closure-shaped future whose captures are its stored fields.


In [ ]:
// 7.1 Calling an async fn constructs; awaiting drives. No runtime yet — the frame is a value.
async fn add(a: i32, b: i32) -> i32 { a + b }
async fn hold_string(s: String) -> usize {
    let n = s.len();          // local BEFORE the await...
    identity_unit().await;    // ...suspension: the frame must hold `n` and `s`
    n                         // ...local AFTER the await, restored from the frame
}
async fn identity_unit() {}

// constructing runs NOTHING: add(2, 3) has not executed a single instruction
let f = add(2, 3);
assert_eq!(std::mem::size_of_val(&f), 12); // two i32 args + the state-tag

// a runtime (next section) polls it to completion; block_on is the smallest one
let rt = tokio::runtime::Runtime::new().unwrap();
assert_eq!(rt.block_on(f), 5);
assert_eq!(rt.block_on(hold_string(String::from("abcdefghij"))), 10);
println!("constructed 2 futures; the runtime polled them to 5 and 10");


In [ ]:
// 7.2 The frame made visible: size is the locals it must keep alive across awaits.
async fn big_frame() -> usize {
    let buf = [0u8; 4096];    // lives across the await -> lives in the future
    identity_unit().await;
    buf.len()
}

// each future holds exactly what its async fn needs across suspension:
let f_add = add(2, 3);                       // two i32s + tag
let f_hold = hold_string(String::from("abcdefghij")); // String + usize + tag
let f_big = big_frame();                     // a 4096-byte array + tag
println!(
    "future sizes: add={} hold_string={} big_frame={} (String itself is {})",
    std::mem::size_of_val(&f_add),
    std::mem::size_of_val(&f_hold),
    std::mem::size_of_val(&f_big),
    std::mem::size_of::<String>(),
);
// the asserts are the point: the state machine is a compile-time-sized VALUE
assert_eq!(std::mem::size_of_val(&f_add), 12);
assert_eq!(std::mem::size_of_val(&f_hold), 64);
assert_eq!(std::mem::size_of_val(&f_big), 4098);


Three reads on those cells. **`size_of_val(&f_add) == 12`** — two `i32` arguments plus the state tag — is the proof that "the future stores the args": no magic, just a struct with two fields. **64 for `hold_string`** is the 24-byte `String`, the `usize`, and padding around the tag: the locals that survive the suspension point are *fields*, allocated at construction — an `async fn` with a 4 KiB array across its await (4098) has a 4 KiB future, and a loop that builds ten thousand such futures has, honestly, ten thousand arrays. The §8 lesson (spawn it) and the §10 lesson (don't hold big buffers across awaits) are both already visible in the numbers. **The `assert_eq!`s stay silent on your machine forever** — the frame layout is fixed at compile time, unlike §6's milliseconds; the chapter teaches you which kind of claim each is.


## 8. The Runtime: `tokio`, `spawn`, `JoinHandle`

A future is inert (§7) — something must call `poll`, remember wakers, and multiplex thousands of suspended frames over a handful of threads. That something is the **runtime**, and there is none in std: `tokio`'s `Runtime` bundles a work-stealing scheduler, N worker threads, an I/O driver (epoll/kqueue/IOCP), and a timer wheel. `Runtime::new()` builds all of it; `rt.block_on(fut)` drives one root future to completion, the calling thread participating. **`tokio::spawn(fut)`** is the scheduler's unit of concurrency: it moves a future onto the runtime as an independent **task** and returns a **`JoinHandle`** — itself a `Future` whose output is `Result<T, JoinError>` (`Err` if the task panicked or was aborted; drop the handle and the task detaches, keep it and you can wait). `spawn` demands the future be **`'static` + `Send`** — owned data, and a state machine that can move between worker threads; capture an `Rc` across an `.await` and `spawn` refuses with the same E0277 as §2, because the *frame* is what crosses threads. `tokio::join!` polls several futures *concurrently on one task* (no spawning) and keeps every result; `tokio::time::sleep` is the timer wheel's primitive, the async world's `thread::sleep` that blocks no thread at all.


In [ ]:
// 8.1 tokio::spawn fans out tasks; JoinHandle awaits a Result; join! runs futures concurrently.
let rt = tokio::runtime::Runtime::new().unwrap();
let total: u32 = rt.block_on(async {
    // four independent tasks — possibly on different worker threads
    let handles: Vec<_> = (0..4)
        .map(|i| tokio::spawn(async move { i * i }))
        .collect();
    let mut sum = 0;
    for h in handles {
        sum += h.await.unwrap(); // JoinHandle: Future<Output = Result<u32, JoinError>>
    }
    // join!: two futures polled concurrently ON THIS TASK, both results kept
    let (a, b) = tokio::join!(async { 20 }, async { 22 });
    assert_eq!(a + b, 42);
    sum
});
assert_eq!(total, 14); // 0 + 1 + 4 + 9 — the SET is certain, the order is not
println!("4 spawned tasks summed to {}; join! kept both results", total);


In [ ]:
// 8.2 The blocking pool and cancellation: spawn_blocking, abort at the next await.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    // CPU-bound or blocking-FFI work goes to the BLOCKING pool, keeping the
    // async workers free to poll — the chapter's first "don't block workers" tool
    let big = tokio::task::spawn_blocking(|| (1..=1_000_000u64).sum::<u64>());
    assert_eq!(big.await.unwrap(), 500_000_500_000);

    // abort: the task dies at its NEXT await point — cancelled, not killed
    let handle = tokio::spawn(async {
        tokio::time::sleep(std::time::Duration::from_secs(60)).await;
        1
    });
    handle.abort();
    assert!(handle.await.is_err()); // JoinError: cancelled
    println!("spawn_blocking summed 1..=1e6 off the workers; abort() cancelled the sleeper");
});


Three reads on those cells. **The `JoinHandle` is the spawn-world's `thread::JoinHandle` with async manners**: awaiting it is `join()`, the `Result` is the task's panic/cancellation story, and dropping it silently detaches — chapter 1's "every thread gets joined" discipline returns as "every task's handle is awaited or deliberately dropped". **`join!` vs `spawn` is the concurrency taxonomy in one cell**: `join!` = two futures interleaved on *one* task (no parallelism, but overlapped waiting — enough for I/O); `spawn` = true multi-worker parallelism; choose per the need, because spawned tasks cost scheduling, `join!`ed ones cost nothing but frame size. **`spawn_blocking` exists because §10's first rule is absolute** — a blocking call on an async worker stalls every task multiplexed on it; the blocking pool is tokio's pressure valve for the work that simply cannot be awaited. And a trap to name now, because the compiler only tells you at runtime: **never drop a `Runtime` from inside async code** — tokio panics with *"Cannot drop a runtime in a context where blocking is not allowed"*; the runtime owns the workers, so it must be created outside and dropped outside any block_on.


## 9. `select!` and `timeout`: Racing Futures

Two futures, one winner: **`tokio::select!`** polls several futures concurrently and runs the body of whichever is **ready first** — the losers are *dropped at their next await*, which is the async world's cancellation: not a kill signal but a state machine (§7) going out of scope. **`tokio::time::timeout(deadline, fut)`** wraps any future: `Ok(output)` if it finished in time, `Err(Elapsed)` if not — and on a miss the future is likewise dropped. Both are the deadline discipline every server loop is made of: *watch the feed, but not forever*. The async channels complete the picture: `tokio::sync::oneshot` is a one-shot reply slot (awaitable `Receiver`), and `tokio::sync::mpsc::channel(cap)` is §3's `mpsc` rebuilt for awaits — with **backpressure**: a bounded channel's `send().await` suspends when the buffer is full, so a slow consumer pushes back on a fast producer without any thread blocking.


In [ ]:
// 9.1 select!: first-ready wins; both arms' bodies must agree on a type.
let rt = tokio::runtime::Runtime::new().unwrap();
let path = rt.block_on(async {
    tokio::select! {
        _ = tokio::time::sleep(std::time::Duration::from_millis(30)) => "slow path",
        _ = tokio::time::sleep(std::time::Duration::from_millis(5)) => "fast path",
    }
});
assert_eq!(path, "fast path");
println!("select! took the arm that finished first: {path}");


In [ ]:
// 9.2 oneshot replies, bounded mpsc with backpressure, timeout as the deadline.
use std::time::Duration;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    // oneshot: exactly one value crosses; the receiver is itself a Future
    let (otx, orx) = tokio::sync::oneshot::channel::<&'static str>();
    tokio::spawn(async move { otx.send("fill arrived").unwrap(); });
    assert_eq!(orx.await.unwrap(), "fill arrived");

    // bounded mpsc: capacity 4; when full, send().await SUSPENDS (backpressure)
    let (tx, mut rx) = tokio::sync::mpsc::channel::<i32>(4);
    tokio::spawn(async move {
        for i in 1..=3 { tx.send(i).await.unwrap(); }
    }); // tx dropped at the end of the task: the channel closes
    let mut got = Vec::new();
    while let Some(v) = rx.recv().await { got.push(v); }
    assert_eq!(got, vec![1, 2, 3]);

    // timeout: Ok in time, Err(Elapsed) on a miss (the future is dropped)
    assert!(tokio::time::timeout(Duration::from_millis(100), tokio::time::sleep(Duration::from_millis(5))).await.is_ok());
    assert!(tokio::time::timeout(Duration::from_millis(5), tokio::time::sleep(Duration::from_millis(100))).await.is_err());
    println!("oneshot + bounded mpsc drained; deadline enforced in both directions");
});


Three reads on those cells. **`select!`'s losing arms are *dropped*, and dropping is cancellation** — the futures' frames (§7) are deallocated mid-flight, so a `select!` arm should not hold resources that need explicit release; when teardown matters, the shutdown signal is a *message* (a channel receive), not a race. **The bounded channel's full-buffer `send().await` is backpressure you get for free**: producer and consumer now move at the speed of the slower one, and no thread ever spins — the std `sync_channel(N)` of §3 was the same idea with blocking, this is the await-flavored twin. **`timeout`'s two asserts are the service-SLO shape**: "the fast upstream answers inside 100ms" and "a pathological 100ms call is cut at 5ms" — `Err(Elapsed)` is not an error to log so much as a *policy decision that fired*, and §18's Redis client will wrap every call in exactly this.


## 10. Blocking vs Async: The Rules of the Road

One decision per function, made deliberately. **Compute-bound, few tasks, or a dependency only offers sync** → threads (§1–§5): real parallelism for real cores. **I/O-bound at any scale, or composing with the async ecosystem** (chapters 17–19) → `async` (§7–§9): cheap concurrency for waiting. The async worker's core covenant: **never block a worker thread** — every task multiplexed on it stalls with you. The rules of the road, each demonstrated in a cell:

| Rule | Tool | Why |
|---|---|---|
| Block? go to the blocking pool | `tokio::task::spawn_blocking` | §8.2 showed the valve; 8 threads by default |
| Wait for sync code from async? | `tokio::task::block_in_place` | converts the current *worker* to a blocker (multi-thread runtime only) |
| Short section, rare, async | `std::sync::Mutex` | fine to hold across **no** await |
| Shared + awaits while holding | `tokio::sync::Mutex` | guard is not `Send` — `lock().await` instead |
| Sub-second sleep from async | `tokio::time::sleep` | never `thread::sleep` on a worker |
| CPU-heavy loop inside async | move it out / `spawn_blocking` | a worker is a shared resource |


In [ ]:
// 10.1 Sync code inside async: brief std Mutex OK; long section -> block_in_place.
use std::sync::Mutex as StdMutex;

let rt = tokio::runtime::Runtime::new().unwrap();
let ledger = Arc::new(StdMutex::new(vec![10u32, 20]));
rt.block_on(async {
    // brief: lock, use, drop BEFORE any await — the guard never crosses an await
    {
        let mut g = ledger.lock().unwrap();
        g.push(30);
    } // released here — never hold across an await
    assert_eq!(ledger.lock().unwrap().len(), 3);

    // a genuinely long sync section: hand the WORKER to the blocking regime
    let ledger2 = Arc::clone(&ledger);
    tokio::task::block_in_place(|| {
        // sync context: plain lock, plain loop, no await in sight
        let mut g = ledger2.lock().unwrap();
        for _ in 0..10 { g[0] += 1; }
    });
    assert_eq!(ledger.lock().unwrap()[0], 20);
    println!("std Mutex held briefly + block_in_place for the long part: workers stayed free");
});


In [ ]:
// 10.2 tokio::sync::Mutex: the guard is not Send, so locking itself is an await.
use std::sync::Arc as StdArc;

let rt = tokio::runtime::Runtime::new().unwrap();
let shared = StdArc::new(tokio::sync::Mutex::new(0u32));
let s2 = StdArc::clone(&shared);
rt.block_on(async move {
    // two tasks incrementing concurrently: the await point is where they interleave
    let h1 = tokio::spawn(async move {
        for _ in 0..2 {
            let mut g = s2.lock().await;
            *g += 1;
        }
    });
    let s3 = StdArc::clone(&shared);
    let h2 = tokio::spawn(async move {
        for _ in 0..2 {
            let mut g = s3.lock().await;
            *g += 1;
        }
    });
    let (a, b) = tokio::join!(h1, h2);
    a.unwrap();
    b.unwrap();
    assert_eq!(*shared.lock().await, 4);
    println!("tokio Mutex: 4 interleaved increments across 2 tasks = {}", shared.lock().await);
});


## 11. Reading the Compiler: E-Codes

| Diagnostic | The story it tells |
|---|---|
| **E0277** | The trait bound failed — and in this chapter it *is* the race detector: `Rc<i32>` cannot be sent between threads safely (needs `Arc`); `RefCell<i32>` cannot be shared between threads safely (needs `Mutex`); a `spawn` future holding `Rc` across an `.await` refuses identically (the *frame* crosses threads). |
| **E0382** | Use of moved value — a `send` moved the message into the channel (§3); the compiler will not let you touch it afterwards. Ownership made the race unrepresentable. |
| **E0597** | Borrowed value does not live long enough — `thread::spawn`'s closure borrows a local that dies first; the fix is `move` + owned data, or `thread::scope` (§2) which extends the borrow to the scope's lifetime *legally*. |
| **E0502** | Cannot borrow as immutable because also borrowed as mutable — inside `thread::scope`, a read overlapping live chunk-borrows; the borrow region ends at the scope's join, not at your manual `join()` calls. |
| **E0599** | No method found — `send` on a `Sender<T>` after it moved into a closure; `lock().await` attempted on a `std::sync::Mutex` guard (that one has no async method); a method on a *guard* that belongs to the guarded type. |
| **E0308** | `select!` arms' body types disagree (§9's rule) — the match arms must unify. |


In [ ]:
// 11.1 E0382 as a compile-time race detector: send moves; use-after-send refuses.
let (tx, rx) = std::sync::mpsc::channel::<String>();
let msg = String::from("fill@100.5x10");
tx.send(msg).unwrap();
// println!("{}", msg);  // error[E0382]: borrow of moved value: `msg`
//                       //   note: partial move... value moved into closure... `send`
assert_eq!(rx.recv().unwrap(), "fill@100.5x10");
println!("send moved the message; the compiler refuses the use-after-send (E0382)");


In [ ]:
// 11.2 E0597/E0502 as scope discipline: borrows inside a scope end AT the scope.
let data = vec![1, 2, 3];
let handle = {
    // thread::spawn(|| data.len()) would be E0597: borrowed value does not live long enough
    let owned = data.clone(); // the move+own fix: nothing borrowed crosses the spawn
    thread::spawn(move || owned.iter().sum::<i32>())
};
assert_eq!(handle.join().unwrap(), 6);
assert_eq!(data, vec![1, 2, 3]);
println!("spawn took owned data; the borrow-refusal (E0597) never had to happen");


Three reads on those cells. **E0382 here is chapter 5's move checker wearing a concurrency badge** — the same error that annoyed you in week one is the reason "two threads mutated the same message" cannot compile in Rust; the channel *is* the ownership transfer, and the error text even names it ("value moved into closure"). **11.2 is the E0597 fix demonstrated rather than suffered**: `move` + owned data (or `Arc`) is the shape that makes the borrow question vanish — and when borrowing genuinely must happen, §2.2's `scope` is the *legal* way to extend the borrow region. The unifying read: **every concurrency E-code in this chapter is an ownership or trait error you already know**, applied to a new boundary — the compile-time race proof is not a new subsystem, it is chapters 5 and 10 aimed at threads.


## 12. Mental Model: The Receipts

| Claim | Receipt |
|---|---|
| A thread is a real, preemptive unit | 1.1: `spawn`→`join` round-trips a value; `Builder` names it; the OS gave us 32 cores |
| `Send`/`Sync` make races compile errors | 2.1: `Arc` crosses, `Rc` is E0277 with the `Arc` fix in the help text; `RefCell` is Send-not-Sync |
| `scope` extends borrows to threads, safely | 2.2: borrowed `chunks_mut` fan-out mutated the vec; reads land only after the scope |
| Channels make races unrepresentable | 3.1: `send` moves; fan-in sorts before asserting the *set* |
| Errors are lifecycle signals | 3.2: `SendError`/`RecvError` = shutdown; `sync_channel(0)` is hand-to-hand |
| The guard is the lock (RAII) | 4.1: 8×100 = 800 through one `Mutex`, no manual unlock anywhere |
| Poisoning is explicit breakage | 4.2: lock returns `Err` after a panic; `into_inner` recovers (and the bind-the-PoisonError deadlock is real — probe-verified) |
| Readers compose, writers exclude | 4.3: two `read()` guards at once, then one `write()` |
| CAS is the lock-free primitive | 5.1: `compare_exchange` loser receives the *current* value in the `Err` |
| Waiting is the workload servers have | 6.1: 10,000 sleepers — threads 261ms vs tasks 33ms (~8×) |
| Futures are compile-time-sized frames | 7.1/7.2: 12, 64, 4098 — the state machine printed in bytes |
| `spawn` = independent task, `join!` = concurrent-on-one-task | 8.1: handles summed 14; `join!` kept both results |
| Cancellation happens at await points | 8.2: `abort()` → `JoinError`; `spawn_blocking` kept workers free |
| `select!` drops the losers | 9.1: first-ready arm ran; 9.2: bounded `send().await` is backpressure |
| One decision per function | 10.1/10.2: brief std-`Mutex` + `block_in_place`; `tokio::sync::Mutex` for across-await |


## 13. Cheat Sheet

```rust
use std::thread;                       // OS threads
thread::spawn(|| ...).join().unwrap(); // spawn + wait; join -> Result<T>
thread::scope(|s| { s.spawn(|| ...); });// borrow non-'static data; auto-join
use std::sync::mpsc;                   // channels
let (tx, rx) = mpsc::channel::<T>();   // many producers, one consumer
tx.send(v).unwrap();                   // moves v; Err(SendError) if rx dropped
for v in rx { ... }                    // ends when all senders drop
let m = Arc::new(Mutex::new(0));       // shared + guarded
*m.lock().unwrap() += 1;               // guard: DerefMut, unlock on drop
m.lock().unwrap_or_else(|p| p.into_inner()) // poison recovery (inspect inline!)
rw.read() / rw.write()                 // RwLock: N readers XOR 1 writer
n.fetch_add(1, Ordering::Relaxed);     // atomic counter (returns PREVIOUS)
cell.compare_exchange(cur, new, SeqCst, SeqCst) // CAS: Err(current) on loss
let rt = tokio::runtime::Runtime::new().unwrap(); // the runtime (notebooks/tests)
rt.block_on(async { fut.await });      // drive one future to completion
tokio::spawn(async { ... }).await      // independent task; JoinHandle->Result
tokio::join!(f1, f2)                   // concurrent on ONE task, keep both
tokio::select! { pat = f1 => ..., pat = f2 => ... } // first-ready wins; losers dropped
tokio::time::timeout(d, fut).await     // Ok(out) | Err(Elapsed); miss = drop
tokio::sync::oneshot::channel()        // single reply slot
tokio::sync::mpsc::channel(cap)        // bounded; send().await = backpressure
tokio::task::spawn_blocking(|| ...).await // CPU/blocking off the async workers
tokio::task::block_in_place(|| ...)    // long sync section on a worker
tokio::sync::Mutex::lock().await       // guard not Send: lock across awaits
```

Seven load-bearing sentences: **a thread per unit of real parallel work, joined; a channel per data flow; a guard per shared datum; an atomic per word of hot state; a future per unit of waiting; a runtime to poll them; `spawn_blocking` for everything that refuses to await.** The sync rules in one line each: *never hold a std guard across an `.await`; never `thread::sleep` on a worker; never assume `select!`-loser cleanup ran; never drop a runtime inside async.*


## 14. Review

Twelve questions, in chapter order. Answers are in the prose above each — none require a machine, though §6's ratio and §7's sizes are worth re-printing from memory.

1. **(§1)** What does `join()` return, and why is it a `Result` rather than the bare value?
2. **(§2)** `Rc<T>` across `thread::spawn` is E0277. Which trait bound fails, what is the standard fix, and why is the failure a *correctness* proof rather than a style complaint?
3. **(§2)** `RefCell<T>` is `Send` but not `Sync`. State the two sentences that distinguish the traits, and what each refusal protects.
4. **(§2)** When is `thread::scope` the right tool instead of `spawn` + owned data, and where do the scope's borrows end?
5. **(§3)** Why does `send` taking ownership make channel races *unrepresentable* rather than merely unlikely?
6. **(§3)** What closes an `mpsc` channel, and which two idioms depend on that closing (`for v in rx` and `recv`)?
7. **(§4)** A panic while holding a `Mutex` poisons it. What do later `lock()` calls return, how do you recover, and which *call-shape* avoids a deadlock while inspecting?
8. **(§4)** When does `RwLock` beat `Mutex`, and what is the exclusivity rule in one sentence?
9. **(§5)** `fetch_add` returns what, and why is `compare_exchange`'s `Err` not really an error?
10. **(§6–§7)** What is a `Future` physically, and why is the size of `add`'s future 12 bytes while `hold_string`'s is 64?
11. **(§8–§9)** `join!` vs `spawn`: which gives parallelism, which gives concurrency-on-one-task, and what does a dropped `JoinHandle` do? What are `select!`'s losing arms, mechanically?
12. **(§10)** Name the four "never" rules of the async road, and the tool for each escape route.

### What Comes Next

Chapter 16's machinery is the pivot the applied track stands on — every remaining chapter drives futures through runtimes.

| Next | What it borrows from here |
|---|---|
| **17 — HTTP Clients and WebSockets** | `reqwest`'s blocking vs async client split is §10 made manifest: same library, two runtimes, and the *live price feed* is `select!` + `timeout` + bounded channels. |
| **18 — Redis** | async connections on the §8 runtime; §9.2's bounded-channel backpressure becomes the pipeline discipline; timeouts wrap every call. |
| **19 — PostgreSQL** | `sqlx` pools are `Arc`-shared guarded state (§4) with `lock().await` semantics; transactions must survive `await` points — §10's rules verbatim. |
| **20 — PyO3** | the escape hatch back to Python: the GIL returns, and with it the *one-thread-at-a-time* world this chapter just left — worth revisiting §6 with that contrast in mind. |

Carry forward: **the await point is the suspension point; the guard drop is the unlock; the dropped future is the cancellation.**
